## Lab 8: Attention

### Objective

To implement and analyze Bahdanau and Luong attention mechanisms in a Seq2Seq Encoder–Decoder model and compare their effectiveness in improving neural machine translation performance.

### Theory

Neural Machine Translation (NMT) is a deep learning approach that automatically translates text from one language to another by learning patterns from bilingual datasets. One of the fundamental architectures used in NMT is the Sequence-to-Sequence (Seq2Seq) model, which consists of an encoder and a decoder. The encoder reads the input sentence and converts it into hidden representations, while the decoder uses this information to generate the translated sentence one word at a time.

In a standard Seq2Seq model, the encoder compresses the entire input sequence into a single fixed-length context vector before passing it to the decoder. Although this approach works reasonably well for short sentences, its performance decreases as sentence length increases because the fixed-size vector cannot preserve all the important information from longer sequences. Consequently, the decoder may lose contextual details, leading to inaccurate translations.
To overcome this limitation, the attention mechanism was introduced. Instead of relying on a single context vector, attention allows the decoder to access all encoder hidden states during each decoding step. It assigns different importance (attention weights) to different input words, enabling the decoder to focus on the most relevant parts of the source sentence while predicting the next target word. This dynamic selection of information significantly improves translation quality, especially for long and complex sentences.
The context vector at each decoding step is computed as:

ct=i=1 ∑Txαt,ihi

where hirepresents the encoder hidden states and αt,i denotes the attention weights that determine the contribution of each hidden state. The attention weights are normalized using the Softmax function so that their values sum to one.

Although the overall concept of attention remains the same, different methods are used to calculate the alignment scores between encoder and decoder states. In this experiment, two widely used attention mechanisms are implemented and compared:

Bahdanau Attention (Additive Attention), which computes alignment scores using a feed-forward neural network.

Luong Attention (Multiplicative Attention), which calculates alignment scores using dot-product similarity, making it computationally more efficient.

Both methods improve the Seq2Seq model by enabling the decoder to retrieve relevant contextual information dynamically instead of depending solely on a single fixed representation of the input sequence.

### 8.1 Bahdanau Attention (Additive Attention)

Bahdanau Attention, also known as Additive Attention, was introduced to address the information bottleneck found in traditional Seq2Seq models. Instead of forcing the encoder to summarize the entire input sentence into a single context vector, this mechanism enables the decoder to determine which encoder hidden states are most relevant for predicting the next output word.

At every decoding step, the previous decoder hidden state is compared with each encoder hidden state using a small feed-forward neural network. This comparison produces alignment scores that measure the relevance of each input word. The scores are then normalized using the Softmax function to obtain attention weights, where larger weights indicate more important encoder states.

### 8.2 Luong Attention (Multiplicative Attention)

Luong Attention, also known as Multiplicative Attention or Dot-Product Attention, is another widely used attention mechanism designed to improve the Seq2Seq architecture. Unlike Bahdanau Attention, which uses a feed-forward neural network to calculate alignment scores, Luong Attention measures the similarity between the current decoder hidden state and the encoder hidden states using matrix multiplication. This simpler computation reduces the computational cost while maintaining high translation performance.

In this approach, the decoder first generates its current hidden state, and then attention is applied to identify the most relevant encoder outputs. The resulting context vector is combined with the decoder hidden state to produce an attentional hidden state, which is finally used to predict the next output word.

### Code

In [7]:
from __future__ import unicode_literals, print_function, division
from io import open
import unicodedata
import re
import random
import os

import torch
import torch.nn as nn
from torch import optim
import torch.nn.functional as F

import numpy as np
from torch.utils.data import TensorDataset, DataLoader, RandomSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
print(f"Using device = {torch.get_default_device()}")

Using device = cpu


In [8]:
SOS_token = 0  # Start of the Sentence
EOS_token = 1  # End of the Sentence

class Lang:
    def __init__(self, name):
        self.name = name
        self.word2index = {}
        self.word2count = {}
        self.index2word = {0: "SOS", 1: "EOS"}
        self.n_words = 2  # Count SOS and EOS

    def addSentence(self, sentence):
        for word in sentence.split(' '):
            self.addWord(word)

    def addWord(self, word):
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.word2count[word] = 1
            self.index2word[self.n_words] = word
            self.n_words += 1
        else:
            self.word2count[word] += 1

In [9]:
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z!?]+", r" ", s)
    return s.strip()

def readLangs(path: str):
    lang1 = 'eng'; lang2 = 'fra'
    print("Reading lines...")
    lines = open(path, encoding='utf-8').read().strip().split('\n')
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    pairs = [list(reversed(p)) for p in pairs]  # French -> English
    input_lang = Lang(lang2)
    output_lang = Lang(lang1)
    return input_lang, output_lang, pairs

In [10]:
MAX_LENGTH = 5

eng_prefixes = (
    "i am ", "i m ",
    "he is", "he s ",
    "she is", "she s ",
    "you are", "you re ",
    "we are", "we re ",
    "they are", "they re "
)

def filterPair(p):
    return (len(p[0].split(' ')) < MAX_LENGTH and
            len(p[1].split(' ')) < MAX_LENGTH and
            p[1].startswith(eng_prefixes))

def filterPairs(pairs):
    return [pair for pair in pairs if filterPair(pair)]

def prepareData(path):
    input_lang, output_lang, pairs = readLangs(path)
    print("Read %s sentence pairs" % len(pairs))
    pairs = filterPairs(pairs)
    print("Trimmed to %s sentence pairs" % len(pairs))
    print("Counting words...")
    for pair in pairs:
        input_lang.addSentence(pair[0])
        output_lang.addSentence(pair[1])
    print("Counted words:")
    print(input_lang.name, input_lang.n_words)
    print(output_lang.name, output_lang.n_words)
    return input_lang, output_lang, pairs

In [11]:
PATH = r'eng-fra.txt'

input_lang, output_lang, pairs = prepareData(PATH)
print(random.choice(pairs))

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
['nous perseverons', 'we re persevering']


In [12]:
def indexesFromSentence(lang, sentence):
    return [lang.word2index[word] for word in sentence.split(' ')]

def tensorFromSentence(lang, sentence):
    indexes = indexesFromSentence(lang, sentence)
    indexes.append(EOS_token)
    return torch.tensor(indexes, dtype=torch.long, device=device).view(1, -1)

def get_dataloader(batch_size):
    input_lang, output_lang, pairs = prepareData(path=PATH)
    n = len(pairs)
    input_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)
    target_ids = np.zeros((n, MAX_LENGTH), dtype=np.int32)

    for idx, (inp, tgt) in enumerate(pairs):
        inp_ids = indexesFromSentence(input_lang, inp)
        tgt_ids = indexesFromSentence(output_lang, tgt)
        inp_ids.append(EOS_token)
        tgt_ids.append(EOS_token)
        input_ids[idx, :len(inp_ids)] = inp_ids
        target_ids[idx, :len(tgt_ids)] = tgt_ids

    train_data = TensorDataset(torch.LongTensor(input_ids).to(device),
                               torch.LongTensor(target_ids).to(device))
    train_sampler = RandomSampler(train_data)
    train_dataloader = DataLoader(train_data, sampler=train_sampler, batch_size=batch_size)
    return input_lang, output_lang, train_dataloader

In [13]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size, dropout_p=0.1):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
        self.embedding = nn.Embedding(input_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, input):
        embedded = self.dropout(self.embedding(input))
        output, hidden = self.rnn(embedded)
        return output, hidden

### Part 1: Bahdanau Attention (Additive)

In [14]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super(BahdanauAttention, self).__init__()
        self.Wa = nn.Linear(hidden_size, hidden_size)  # W_1: projects encoder states
        self.Ua = nn.Linear(hidden_size, hidden_size)  # W_2: projects decoder query
        self.Va = nn.Linear(hidden_size, 1)             # v^T: scalar scoring

    def forward(self, query, keys):
        """
        query : decoder hidden state s_{t-1}
                shape -> (batch_size, 1, hidden_size)
        keys  : all encoder hidden states h_1,...,h_T
                shape -> (batch_size, seq_len, hidden_size)

        Returns:
            context : weighted sum of encoder states
                      shape -> (batch_size, 1, hidden_size)
            weights : attention weights alpha
                      shape -> (batch_size, 1, seq_len)
        """
        # e_{t,i} = v^T tanh(W_1 h_i + W_2 s_{t-1})
        scores = self.Va(torch.tanh(self.Wa(query) + self.Ua(keys)))
        scores = scores.squeeze(2).unsqueeze(1)          # (batch, 1, seq_len)

        weights = F.softmax(scores, dim=-1)              # alpha_{t,i}
        context = torch.bmm(weights, keys)               # c_t

        return context, weights

### `AttnDecoderRNN` (Bahdanau)

In [15]:
class BahdanauAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(BahdanauAttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.attention = BahdanauAttention(hidden_size)
        self.rnn = nn.RNN(2 * hidden_size, hidden_size, batch_first=True)  # input = embed + context
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                # Teacher forcing: use ground-truth target as next input
                decoder_input = target_tensor[:, i].unsqueeze(1)
            else:
                # Inference: use model's own prediction
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))           # (batch, 1, hidden)

        query = hidden.permute(1, 0, 2)                          # (batch, 1, hidden)
        context, attn_weights = self.attention(query, encoder_outputs)

        input_rnn = torch.cat((embedded, context), dim=2)        # (batch, 1, 2*hidden)
        output, hidden = self.rnn(input_rnn, hidden)
        output = self.out(output)

        return output, hidden, attn_weights

### Part 2: Luong Attention (Multiplicative / Dot-Product)

In [16]:
class LuongDotAttention(nn.Module):
    def __init__(self, hidden_size):
        super(LuongDotAttention, self).__init__()
        # W_c for: s~_t = tanh(W_c [c_t ; s_t])
        self.Wc = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, query, keys):
        """
        query : current decoder hidden state s_t
                shape -> (batch_size, 1, hidden_size)
        keys  : encoder hidden states h_1,...,h_T
                shape -> (batch_size, seq_len, hidden_size)

        Returns:
            attentional_hidden : s~_t
                                 shape -> (batch_size, 1, hidden_size)
            weights            : attention weights alpha_t
                                 shape -> (batch_size, 1, seq_len)
        """
        # Alignment scores: e_{t,i} = s_t^T h_i
        scores = torch.bmm(query, keys.transpose(1, 2))          # (batch, 1, seq_len)

        # Attention weights: alpha_{t,i} = softmax(e_{t,i})
        weights = F.softmax(scores, dim=-1)

        # Context vector: c_t = sum(alpha_{t,i} * h_i)
        context = torch.bmm(weights, keys)                       # (batch, 1, hidden)

        # Attentional hidden state: s~_t = tanh(W_c [c_t ; s_t])
        combined = torch.cat((context, query), dim=-1)           # (batch, 1, 2*hidden)
        attentional_hidden = torch.tanh(self.Wc(combined))       # (batch, 1, hidden)

        return attentional_hidden, weights

### `LuongAttnDecoderRNN`

In [17]:
class LuongAttnDecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size, dropout_p=0.1):
        super(LuongAttnDecoderRNN, self).__init__()
        self.embedding = nn.Embedding(output_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)  # input = embed only
        self.attention = LuongDotAttention(hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        self.dropout = nn.Dropout(dropout_p)

    def forward(self, encoder_outputs, encoder_hidden, target_tensor=None):
        batch_size = encoder_outputs.size(0)
        decoder_input = torch.empty(batch_size, 1, dtype=torch.long, device=device).fill_(SOS_token)
        decoder_hidden = encoder_hidden
        decoder_outputs = []
        attentions = []

        for i in range(MAX_LENGTH):
            decoder_output, decoder_hidden, attn_weights = self.forward_step(
                decoder_input, decoder_hidden, encoder_outputs
            )
            decoder_outputs.append(decoder_output)
            attentions.append(attn_weights)

            if target_tensor is not None:
                decoder_input = target_tensor[:, i].unsqueeze(1)  # Teacher forcing
            else:
                _, topi = decoder_output.topk(1)
                decoder_input = topi.squeeze(-1).detach()

        decoder_outputs = torch.cat(decoder_outputs, dim=1)
        decoder_outputs = F.log_softmax(decoder_outputs, dim=-1)
        attentions = torch.cat(attentions, dim=1)

        return decoder_outputs, decoder_hidden, attentions

    def forward_step(self, input, hidden, encoder_outputs):
        embedded = self.dropout(self.embedding(input))           # (batch, 1, hidden)

        # 1. Run RNN to get current decoder state s_t
        rnn_output, hidden = self.rnn(embedded, hidden)          # (batch, 1, hidden)

        # 2. Compute attention using s_t as query
        query = rnn_output                                        # (batch, 1, hidden)
        attentional_hidden, attn_weights = self.attention(query, encoder_outputs)

        # 3. Predict from attentional hidden state s~_t
        output = self.out(attentional_hidden)                    # (batch, 1, output_size)

        return output, hidden, attn_weights

### Training Infrastructure


In [18]:
import time
import math
import matplotlib.pyplot as plt
plt.switch_backend('agg')
import matplotlib.ticker as ticker

def asMinutes(s):
    m = math.floor(s / 60)
    s -= m * 60
    return '%dm %ds' % (m, s)

def timeSince(since, percent):
    now = time.time()
    s = now - since
    es = s / (percent)
    rs = es - s
    return '%s (- %s)' % (asMinutes(s), asMinutes(rs))

def showPlot(points, title='Training Loss'):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    ax.set_title(title)
    ax.set_xlabel('Epochs (x plot_every)')
    ax.set_ylabel('Average NLL Loss')
    plt.plot(points)
    plt.tight_layout()
    plt.savefig(title.replace(' ', '_') + '.png', dpi=100)
    plt.show()

In [19]:
def train_epoch(dataloader, encoder, decoder, encoder_optimizer,
                decoder_optimizer, criterion):
    total_loss = 0
    for data in dataloader:
        input_tensor, target_tensor = data

        encoder_optimizer.zero_grad()
        decoder_optimizer.zero_grad()

        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, _, _ = decoder(encoder_outputs, encoder_hidden, target_tensor)

        loss = criterion(
            decoder_outputs.view(-1, decoder_outputs.size(-1)),
            target_tensor.view(-1)
        )
        loss.backward()

        encoder_optimizer.step()
        decoder_optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


def train(train_dataloader, encoder, decoder, n_epochs, learning_rate=0.001,
          print_every=100, plot_every=100, title='Training Loss'):
    start = time.time()
    plot_losses = []
    print_loss_total = 0
    plot_loss_total = 0

    encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate)
    criterion = nn.NLLLoss()

    for epoch in range(1, n_epochs + 1):
        loss = train_epoch(train_dataloader, encoder, decoder,
                           encoder_optimizer, decoder_optimizer, criterion)
        print_loss_total += loss
        plot_loss_total += loss

        if epoch % print_every == 0:
            print_loss_avg = print_loss_total / print_every
            print_loss_total = 0
            print('%s (%d %d%%) %.4f' % (timeSince(start, epoch / n_epochs),
                                          epoch, epoch / n_epochs * 100, print_loss_avg))

        if epoch % plot_every == 0:
            plot_loss_avg = plot_loss_total / plot_every
            plot_losses.append(plot_loss_avg)
            plot_loss_total = 0

    showPlot(plot_losses, title=title)

## Evaluation Code

In [20]:
def evaluate(encoder, decoder, sentence, input_lang, output_lang):
    with torch.no_grad():
        input_tensor = tensorFromSentence(input_lang, sentence)
        encoder_outputs, encoder_hidden = encoder(input_tensor)
        decoder_outputs, decoder_hidden, decoder_attn = decoder(encoder_outputs, encoder_hidden)

        _, topi = decoder_outputs.topk(1)
        decoded_ids = topi.squeeze()

        decoded_words = []
        for idx in decoded_ids:
            if idx.item() == EOS_token:
                decoded_words.append('<EOS>')
                break
            decoded_words.append(output_lang.index2word[idx.item()])
    return decoded_words, decoder_attn


def evaluateRandomly(encoder, decoder, n=10):
    for i in range(n):
        pair = random.choice(pairs)
        print('>', pair[0])
        print('=', pair[1])
        output_words, _ = evaluate(encoder, decoder, pair[0], input_lang, output_lang)
        output_sentence = ' '.join(output_words)
        print('<', output_sentence)
        print('')

### Training  Bahdanau Attention

In [21]:
hidden_size = 128
batch_size = 32
EPOCHS = 200

input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

bahdanau_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
bahdanau_decoder = BahdanauAttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, bahdanau_encoder, bahdanau_decoder, EPOCHS,
      print_every=5, plot_every=5, title='Bahdanau Attention - Training Loss')

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 3s (- 2m 17s) (5 2%) 1.8045
0m 6s (- 1m 56s) (10 5%) 1.0298
0m 8s (- 1m 47s) (15 7%) 0.6910
0m 11s (- 1m 42s) (20 10%) 0.4338
0m 14s (- 1m 38s) (25 12%) 0.2695
0m 16s (- 1m 33s) (30 15%) 0.1711
0m 19s (- 1m 29s) (35 17%) 0.1210
0m 21s (- 1m 26s) (40 20%) 0.0929
0m 24s (- 1m 23s) (45 22%) 0.0780
0m 26s (- 1m 20s) (50 25%) 0.0676
0m 29s (- 1m 17s) (55 27%) 0.0626
0m 31s (- 1m 14s) (60 30%) 0.0574
0m 34s (- 1m 11s) (65 32%) 0.0566
0m 36s (- 1m 8s) (70 35%) 0.0514
0m 39s (- 1m 5s) (75 37%) 0.0515
0m 41s (- 1m 2s) (80 40%) 0.0487
0m 44s (- 0m 59s) (85 42%) 0.0507
0m 46s (- 0m 57s) (90 45%) 0.0472
0m 49s (- 0m 54s) (95 47%) 0.0461
0m 51s (- 0m 51s) (100 50%) 0.0458
0m 54s (- 0m 48s) (105 52%) 0.0450
0m 56s (- 0m 46s) (110 55%) 0.0449
0m 58s (- 0m 43s) (115 57%) 0.0449
1m 1s (- 0m 40s) (120 60%) 0.0442
1m 3s (- 0m 38s) (125 62%) 0.0441
1m 6s (- 0m 35s) (130 65%) 0.04

/var/folders/7s/5gnsvf394r55wlyjssk82y840000gn/T/ipykernel_45602/37737821.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Bahdanau — Evaluation

In [22]:
bahdanau_encoder.eval()
bahdanau_decoder.eval()
evaluateRandomly(bahdanau_encoder, bahdanau_decoder)

> tu caches quelque chose
= you re hiding something
< you re hiding something <EOS>

> je suis tellement embarrasse
= i m so embarrassed
< i m so embarrassed <EOS>

> il est canadien
= he is canadian
< he is canadian <EOS>

> nous nous amusons
= we re having fun
< we re enjoying ourselves <EOS>

> je suis simplement fatigue
= i m just tired
< i m just tired <EOS>

> vous etes fort elegante
= you re very sophisticated
< you re very stylish <EOS>

> je suis sur
= i am sure
< i m certain <EOS>

> on est enfin seuls
= we re finally alone
< we re finally alone <EOS>

> tu es brillant
= you re bright
< you re bright something <EOS>

> je suis laid
= i m ugly
< i m ugly <EOS>



---

## Training — Luong Attention

In [23]:
input_lang, output_lang, train_dataloader = get_dataloader(batch_size)

luong_encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
luong_decoder = LuongAttnDecoderRNN(hidden_size, output_lang.n_words).to(device)

train(train_dataloader, luong_encoder, luong_decoder, EPOCHS,
      print_every=5, plot_every=5, title='Luong Attention - Training Loss')

Reading lines...
Read 135842 sentence pairs
Trimmed to 3272 sentence pairs
Counting words...
Counted words:
fra 1757
eng 967
0m 2s (- 1m 32s) (5 2%) 1.8042
0m 4s (- 1m 31s) (10 5%) 1.1062
0m 7s (- 1m 29s) (15 7%) 0.8252
0m 9s (- 1m 26s) (20 10%) 0.6195
0m 12s (- 1m 24s) (25 12%) 0.4617
0m 14s (- 1m 22s) (30 15%) 0.3515
0m 17s (- 1m 20s) (35 17%) 0.2756
0m 19s (- 1m 18s) (40 20%) 0.2186
0m 22s (- 1m 15s) (45 22%) 0.1902
0m 24s (- 1m 13s) (50 25%) 0.1606
0m 26s (- 1m 10s) (55 27%) 0.1454
0m 29s (- 1m 7s) (60 30%) 0.1308
0m 31s (- 1m 5s) (65 32%) 0.1210
0m 33s (- 1m 3s) (70 35%) 0.1150
0m 36s (- 1m 0s) (75 37%) 0.1059
0m 38s (- 0m 58s) (80 40%) 0.1022
0m 41s (- 0m 55s) (85 42%) 0.0965
0m 43s (- 0m 53s) (90 45%) 0.0928
0m 45s (- 0m 50s) (95 47%) 0.0885
0m 48s (- 0m 48s) (100 50%) 0.0852
0m 50s (- 0m 45s) (105 52%) 0.0855
0m 52s (- 0m 43s) (110 55%) 0.0824
0m 55s (- 0m 40s) (115 57%) 0.0807
0m 57s (- 0m 38s) (120 60%) 0.0810
1m 0s (- 0m 36s) (125 62%) 0.0749
1m 2s (- 0m 33s) (130 65%) 0.076

/var/folders/7s/5gnsvf394r55wlyjssk82y840000gn/T/ipykernel_45602/37737821.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [24]:
luong_encoder.eval()
luong_decoder.eval()
evaluateRandomly(luong_encoder, luong_decoder)

> elle est chancarde
= she is lucky
< she is gaining weight <EOS>

> nous sommes tres serieuses
= we re very serious
< we re very serious <EOS>

> c est mon collegue
= he is my colleague
< he is my colleague <EOS>

> nous sommes sinceres
= we re sincere
< we re the problem <EOS>

> je suis gras
= i m fat
< i am extremely fat <EOS>

> c est un lambin
= he s a slowpoke
< he s a slowpoke <EOS>

> elle prend du poids
= she is gaining weight
< she is gaining weight <EOS>

> tu es tres raffine
= you re very sophisticated
< you re very sophisticated <EOS>

> vous etes maigrichon
= you re skinny
< you re skinny <EOS>

> nous petit dejeunons
= we are having breakfast
< we are having friends <EOS>



### Results

Both Bahdanau and Luong attention models were successfully trained and evaluated on the English–French translation dataset. The training loss decreased steadily throughout the training process, indicating effective learning. The generated translations were generally accurate and showed improved contextual understanding compared to a basic Seq2Seq model. Luong Attention trained slightly faster due to its simpler dot-product computation, while Bahdanau Attention also produced reliable translation results.

### Discussion

The implementation of attention mechanisms significantly enhanced the Seq2Seq model by allowing the decoder to focus on relevant encoder hidden states during translation. Bahdanau Attention uses a feed-forward network to compute alignment scores, whereas Luong Attention employs a dot-product approach that is computationally more efficient. Although both methods achieved comparable translation quality, Luong Attention required less computation and trained more quickly. Overall, attention mechanisms improved the model's ability to preserve contextual information, especially for longer input sequences.

### Conclusion

This laboratory demonstrated the successful implementation of Bahdanau and Luong attention mechanisms in a Seq2Seq neural machine translation model. Both approaches improved translation performance by enabling the decoder to dynamically attend to important parts of the input sequence. Among the two, Luong Attention provided slightly better computational efficiency, while both methods produced accurate and context-aware translations, highlighting the importance of attention mechanisms in modern NLP applications.